In [2]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pdfmuse import PdfmuseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0)

docs = PdfmuseLoader("data/sampled_schemes.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0).split_documents(docs)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print(len(chunks), "chunks")




88 chunks


In [3]:
print(chunks[34].page_content)

NeGP takes a holistic view of e­governance initiatives across the country, integrating  them  into  a  collective  vision,  a  shared  cause.  Around  this  idea,  a  massive  countrywide infrastructure reaching down to the remotest of villages is evolving, and


In [4]:
print(chunks[34].metadata)

{'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 3, 'total_pages': 9, 'chunk_id': 34}


In [5]:
from langchain_chroma import Chroma

chroma_store = Chroma(
    collection_name="hdfc_schemes",
    embedding_function=embeddings,
    persist_directory="chroma_db2")


In [6]:
chroma_store.add_documents(chunks)


['3c47184a-122a-442a-9ba4-b6d1ed04329d',
 '2b39716b-1f65-4223-b424-f7b0a2999c55',
 '6da3e289-6374-40e8-a935-7c220e4ad6c0',
 '6c742fc1-edbf-470d-9b89-a1438b7467a9',
 'da8a4019-cd5a-4162-ae28-c873ef8d8882',
 '35b2e1d2-5515-4590-9db5-7099608a7884',
 'cde66f06-2ce3-49a2-a2c2-ae30343fda08',
 'b7004da0-6914-4fc6-a334-7e27847da430',
 '13642e97-1a13-4142-9526-ab77cf2c6f6a',
 '4c9a0622-9ae3-4d68-950f-47da2bb9aff1',
 '95a2956d-ae16-4071-b721-e515d504bdc8',
 'd3508973-f1b9-42ee-a752-b83a41b67f72',
 'f8126578-5f53-4e32-9725-bf745a62224d',
 '99039a56-d277-474c-abb7-74bec5623424',
 '09431735-f108-416e-af50-6f2c967f2020',
 '697ac75d-d24a-4be7-8b45-83081ac56748',
 '49193e9e-f5dc-4844-a726-c51f68bca38d',
 '32360580-32e6-44b5-8766-928b241d1bf8',
 '29f9cd17-b041-4087-9df0-b015a9bc59ea',
 '22173828-4a71-4cf7-a380-e1b12351c870',
 'db519da2-2e2c-466b-aed7-91d656b8e96b',
 '63a9d9b7-3202-40cc-b3be-5cd9e0e87379',
 '2f3263d5-3e9d-4724-8710-4f8215d6d2ce',
 '5b3c94a0-794a-4579-9105-4deb9aa65bda',
 '1c5b5999-a237-

In [7]:
print("Chroma collection count:", chroma_store._collection.count())


Chroma collection count: 88


In [8]:
EVAL_QUERIES = [
    {
        "query": "SMAM",
        "correct": {22},
        "type": "keyword",
    },
    {
        "query": "PKVY",
        "correct": {12, 14, 16},
        "type": "keyword",
    },
    {
        "query": "How does the government make sure a farmer's information is submitted online "
                  "before they can receive the insurance subsidy payout?",
        "correct": {63},
        "type": "paraphrase",
    },
    {
        "query": "What chemical-free, low-cost farming methods does the scheme promote?",
        "correct": {15, 16},
        "type": "paraphrase",
    },
    {
        "query": "What is the objective of the National Food Security Mission?",
        "correct": {4},
        "type": "baseline",
    },
]


# Dense/ Semantic retrieval evaluation

In [9]:

similar_docs = chroma_store.similarity_search("How to download the requisite information",k=2)


In [10]:
results = chroma_store.similarity_search_with_score(
    "How to download the requisite information?", k=3)
results

[(Document(id='3e106044-c120-4340-9ae8-1c2d38edb798', metadata={'source': 'data/sampled_schemes.pdf', 'page': 6, 'total_pages': 9, 'source_kind': 'Pdf', 'chunk_id': 60}, page_content='them\xa0to\xa0enter/upload/download\xa0the\xa0requisite\xa0information.'),
  0.747603714466095),
 (Document(id='f842232c-f23a-4348-a4ed-e8c5a373747c', metadata={'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'total_pages': 9, 'page': 6, 'chunk_id': 66}, page_content='information/data.'),
  1.17890202999115),
 (Document(id='1f65b29f-bf4f-4e89-b05a-ec6c64fc39cb', metadata={'page': 6, 'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'total_pages': 9, 'chunk_id': 57}, page_content='2.6 \n\n2.7 \n\n2.8 \n\n2.9 \n\n2.10 \n\n2.11 \n\n2.12 \n\n\xa0\n\nministrationandseamless\xa0flow\xa0of\xa0data/information/reports\xa0on\xa0real\xa0time\xa0 basis,\xa0 State\xa0 Govt. would\xa0 not\xa0 be\xa0 allowed\xa0 to\xa0 create/use\xa0 separate\xa0 Portal/website\xa0for\xa0Crop\xa0Insurance\xa0purpo

In [11]:
retriever = chroma_store.as_retriever(search_kwargs={"k": 5})
retriever.invoke("Stealing from the bank is a crime")

[Document(id='4ce943cb-f59a-432c-8155-438091797e7a', metadata={'total_pages': 9, 'chunk_id': 81, 'source': 'data/sampled_schemes.pdf', 'source_kind': 'Pdf', 'page': 8}, page_content='Merely,\xa0 sanctioning\xa0 of\xa0 crop\xa0 loan\xa0 against\xa0 other\xa0 collateral\xa0 securities\xa0 including\xa0fixed\xa0deposits,\xa0gold/jewel\xa0loans,\xa0mortgage\xa0loans\xa0etc.\xa0without\xa0 having\xa0insurable\xa0interest\xa0of\xa0the\xa0farmer\xa0on\xa0the\xa0insurable\xa0land\xa0and\xa0notified\xa0 crops\xa0shall\xa0not\xa0be\xa0covered\xa0under\xa0the\xa0Scheme.'),
 Document(id='43da8eb1-dda8-4695-9e51-bd8cecfc87c4', metadata={'chunk_id': 65, 'source': 'data/sampled_schemes.pdf', 'page': 6, 'source_kind': 'Pdf', 'total_pages': 9}, page_content='Banks/Financial\xa0Institutions/other\xa0intermediaries\xa0need\xa0to\xa0compulsorily\xa0 transfer the individual farmer’s\xa0data\xa0electronically\xa0to\xa0the\xa0National\xa0Crop\xa0 Insurance\xa0Portal.\xa0Accordingly\xa0Banks/FIs\xa0may\xa0end

In [12]:
retriever = chroma_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10}
)
retriever.invoke("Stealing from the bank is a crime")

[Document(id='4ce943cb-f59a-432c-8155-438091797e7a', metadata={'total_pages': 9, 'page': 8, 'source_kind': 'Pdf', 'source': 'data/sampled_schemes.pdf', 'chunk_id': 81}, page_content='Merely,\xa0 sanctioning\xa0 of\xa0 crop\xa0 loan\xa0 against\xa0 other\xa0 collateral\xa0 securities\xa0 including\xa0fixed\xa0deposits,\xa0gold/jewel\xa0loans,\xa0mortgage\xa0loans\xa0etc.\xa0without\xa0 having\xa0insurable\xa0interest\xa0of\xa0the\xa0farmer\xa0on\xa0the\xa0insurable\xa0land\xa0and\xa0notified\xa0 crops\xa0shall\xa0not\xa0be\xa0covered\xa0under\xa0the\xa0Scheme.'),
 Document(id='b3f0289b-d676-4278-9d52-cf9383af84d5', metadata={'chunk_id': 54, 'source': 'data/sampled_schemes.pdf', 'page': 5, 'source_kind': 'Pdf', 'total_pages': 9}, page_content='for\xa0any\xa0incorrect\xa0entry/\xa0errors/\xa0omissions\xa0etc.'),
 Document(id='3ef0bf20-88d9-410c-b591-6ce88d6981bd', metadata={'source_kind': 'Pdf', 'chunk_id': 51, 'total_pages': 9, 'source': 'data/sampled_schemes.pdf', 'page': 5}, page_conte

In [13]:
EVAL_QUERIES = [
    {
        "query": "SMAM",
        "correct": {22},
        "type": "keyword",
    },
    {
        "query": "PKVY",
        "correct": {12, 14, 16},
        "type": "keyword",
    },
    {
        "query": "How does the government make sure a farmer's information is submitted online "
                  "before they can receive the insurance subsidy payout?",
        "correct": {63},
        "type": "paraphrase",
    },
    {
        "query": "What chemical-free, low-cost farming methods does the scheme promote?",
        "correct": {15, 16},
        "type": "paraphrase",
    },
    {
        "query": "What is the objective of the National Food Security Mission?",
        "correct": {4},
        "type": "baseline",
    },
]


In [14]:
dense_retriever = chroma_store.as_retriever(search_kwargs={"k": 3})

for item in EVAL_QUERIES:
    results = dense_retriever.invoke(item["query"])
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")

[ HIT] SMAM                                                         -> [22, 38, 0]
[ HIT] PKVY                                                         -> [14, 12, 16]
[ HIT] How does the government make sure a farmer's information is  -> [52, 63, 53]
[ HIT] What chemical-free, low-cost farming methods does the scheme -> [15, 8, 6]
[ HIT] What is the objective of the National Food Security Mission? -> [4, 0, 11]


# Similarity search vs MMR (single query examples)

In [15]:
query = "What are the eligibility criteria for getting an insurance subsidy?"

# Plain similarity search -- pure vector distance, no diversity control
similarity_results = chroma_store.similarity_search(query, k=5)
print([d.metadata["chunk_id"] for d in similarity_results])

[63, 74, 82, 52, 84]


In [16]:
print(chunks[63].page_content)

Only farmers whose data is uploaded on the National Crop Insurance  Portal shall be eligible for Insurance coverage and the premium subsidy  from State and Central Govt. will be released accordingly.


In [17]:
# MMR search -- fetches a wider candidate pool (fetch_k) then re-ranks for
# relevance + diversity (lambda_mult closer to 1 = more relevance, closer to 0 = more diversity)
mmr_results = chroma_store.max_marginal_relevance_search(
    query, k=5, fetch_k=10, lambda_mult=0.5
)
print([d.metadata["chunk_id"] for d in mmr_results])

[63, 74, 84, 24, 57]


# Sparse/keyword retrieval evaluation

In [18]:
#Not Exact keyword search

In [19]:
"What are the eligibility criteria for getting an insurance subsidy?"

'What are the eligibility criteria for getting an insurance subsidy?'

In [20]:
from langchain_community.retrievers import BM25Retriever
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())

naive_bm25 = BM25Retriever.from_documents(chunks,preprocess_func=tokenize)  # default preprocess_func = text.split()
naive_bm25.k = 3

results = naive_bm25.invoke("SMAM")
print([d.metadata["chunk_id"] for d in results])

[22, 87, 85]


In [21]:
chunks[28].metadata

{'source': 'data/sampled_schemes.pdf',
 'source_kind': 'Pdf',
 'page': 3,
 'total_pages': 9,
 'chunk_id': 28}

In [22]:
print(chunks[30].page_content)

Intervention: Gains made in the past will be consolidated and strengthened through  increased penetration of extension functionaries. Personnel trained under Agri­Clinics  and  Agri­Business  Centres  Scheme  (ACABC)  and  Diploma  in  Agriculture  Extension


In [23]:
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())

bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
bm25_retriever.k = 3

for item in EVAL_QUERIES:
    results = bm25_retriever.invoke(item["query"])
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")


[ HIT] SMAM                                                         -> [22, 87, 85]
[ HIT] PKVY                                                         -> [16, 14, 12]
[miss] How does the government make sure a farmer's information is  -> [61, 36, 65]
[ HIT] What chemical-free, low-cost farming methods does the scheme -> [31, 15, 45]
[ HIT] What is the objective of the National Food Security Mission? -> [0, 4, 22]


In [24]:
print(chunks[61].page_content)

Insurance  Companies  shall  not  distribute/collect/allow  any  other  proforma /utility/web Portal etc. for collecting details of insured farmers  separately. However they may provide all requisite support to facilitate  Bank Branches/ PACS for uploading the farmer’s details on the Portal


# Hybrid search (Advance reterival strategy)

In [25]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever], weights=[0.5, 0.5]
)

for item in EVAL_QUERIES:
    results = hybrid_retriever.invoke(item["query"])[:3]
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")


[ HIT] SMAM                                                         -> [22, 38, 87]
[ HIT] PKVY                                                         -> [14, 16, 12]
[ HIT] How does the government make sure a farmer's information is  -> [52, 61, 63]
[ HIT] What chemical-free, low-cost farming methods does the scheme -> [15, 31, 8]
[ HIT] What is the objective of the National Food Security Mission? -> [4, 0, 11]


# Re-ranking

In [26]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

wide_dense = chroma_store.as_retriever(search_kwargs={"k": 8})
wide_bm25 = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
wide_bm25.k = 8
wide_hybrid_retriever = EnsembleRetriever(retrievers=[wide_dense, wide_bm25], weights=[0.5, 0.5])

reranker = FlashrankRerank(top_n=3)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker, base_retriever=wide_hybrid_retriever
)

for item in EVAL_QUERIES:
    results = rerank_retriever.invoke(item["query"])
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")


INFO:flashrank.Ranker:Downloading ms-marco-MultiBERT-L-12...
ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:17<00:00, 5.84MiB/s]
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] SMAM                                                         -> [22, 28, 38]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] PKVY                                                         -> [16, 12, 14]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] How does the government make sure a farmer's information is  -> [52, 53, 63]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[miss] What chemical-free, low-cost farming methods does the scheme -> [31, 71, 12]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[miss] What is the objective of the National Food Security Mission? -> [1, 22, 21]


# Multi-Query Retrieval

In [27]:
from langchain_classic.retrievers import MultiQueryRetriever

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever,
    llm=llm
)

query = "What are the eligibility criteria for getting an insurance subsidy?"

results = multiquery_retriever.invoke(query)

print([d.metadata["chunk_id"] for d in results])

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What requirements must an individual meet to qualify for an insurance subsidy?', 'Who is eligible to receive financial assistance or premium subsidies for insurance coverage?', 'What income, household, residency, or coverage conditions determine eligibility for an insurance subsidy?']
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[63, 74, 52, 73, 82]


In [28]:
results[0].metadata

{'source_kind': 'Pdf',
 'source': 'data/sampled_schemes.pdf',
 'chunk_id': 63,
 'total_pages': 9,
 'page': 6}

In [34]:
print(results[4].page_content)

Voluntary Component    The Scheme is optional for non­loanee farmers.    The  insurance  coverage  will  strictly  be  equivalent  to  sum  insured  /hectare, as defined in the Govt. notification or/and on National Crop  Insurance Portal multiplied by sown area for notified crop.


In [35]:
from langchain_classic.retrievers import MultiQueryRetriever

multiquery_retriever = MultiQueryRetriever.from_llm(retriever=dense_retriever, llm=llm)

mismatch_query = EVAL_QUERIES[2]["query"]  # the online-submission / insurance-subsidy paraphrase
results = multiquery_retriever.invoke(mismatch_query)
print([d.metadata["chunk_id"] for d in results])


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What government process verifies that a farmer’s details have been uploaded online before releasing an insurance subsidy payment?', 'How is online submission of farmer information made a prerequisite for receiving the crop insurance subsidy payout?', 'What checks or digital verification steps ensure a farmer’s data is submitted through the government portal before the insurance subsidy is disbursed?']
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[52, 63, 61, 65]


In [37]:
print(results[3].page_content)

Banks/Financial Institutions/other intermediaries need to compulsorily  transfer the individual farmer’s data electronically to the National Crop  Insurance Portal. Accordingly Banks/FIs may endeavour to undertake  CBS  integration  in  a  time  bound  manner  for  real  time  transfer  of


In [38]:
from langchain_core.prompts import ChatPromptTemplate

HYDE_PROMPT = ChatPromptTemplate.from_template(
    "Write a short passage (2-3 sentences) that would answer this question, in the style of an "
    "Indian government scheme document. State the answer directly, as fact -- do not hedge or "
    "say you're unsure.\n\nQuestion: {question}"
)
hyde_chain = HYDE_PROMPT | llm

def hyde_retrieve(query, k=3):
    hypothetical_doc = hyde_chain.invoke({"question": query}).content
    return chroma_store.similarity_search(hypothetical_doc, k=k)

for item in EVAL_QUERIES:
    results = hyde_retrieve(item["query"])
    found_ids = [d.metadata["chunk_id"] for d in results]
    hit = bool(item["correct"] & set(found_ids))
    print(f"[{'HIT' if hit else 'miss':>4}] {item['query'][:60]:60} -> {found_ids}")


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] SMAM                                                         -> [22, 23, 21]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] PKVY                                                         -> [12, 16, 14]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] How does the government make sure a farmer's information is  -> [52, 63, 61]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] What chemical-free, low-cost farming methods does the scheme -> [15, 18, 8]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[ HIT] What is the objective of the National Food Security Mission? -> [4, 2, 11]


In [39]:
query = "What are the eligibility criteria for getting an insurance subsidy?"

similarity_results = chroma_store.similarity_search(query, k=5)
print("similarity:", [d.metadata["chunk_id"] for d in similarity_results])

mmr_results = chroma_store.max_marginal_relevance_search(
    query, k=5, fetch_k=10, lambda_mult=0.5
)
print("mmr:", [d.metadata["chunk_id"] for d in mmr_results])

INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


similarity: [63, 74, 82, 52, 84]


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


mmr: [63, 74, 84, 24, 57]
